# Phase 1 – Data Cleaning Pipeline
Wide → Long conversion, threshold validation, outlier detection (rolling MAD), and imputation.

In [11]:
import pandas as pd
import numpy as np
import gc

## 1. Load raw data

In [12]:
input_path = "/Users/phasurab/Desktop/Alto_test/phase_1_dataset.csv"
save_path = "/Users/phasurab/Desktop/Alto_test/final_cleaned_phase1.csv"

raw = pd.read_csv(input_path, low_memory=True)
raw.head()

/var/folders/tl/_94g3_xj39s_4l8ztr4q5hvh0000gn/T/ipykernel_44909/3705230398.py:4: DtypeWarning: Columns (0: room_1001_bedroom, 1: room_1001_bedroom.1, 2: room_1001_bedroom.2, 3: room_1002_bedroom, 4: room_1002_bedroom.1, 5: room_1002_bedroom.2, 6: room_1002_bedroom.3, 7: room_1002_bedroom.4, 8: room_1003_bedroom, 9: room_1003_bedroom.1, 10: room_1003_bedroom.2, 11: room_1003_bedroom.3, 12: room_1003_bedroom.4, 13: room_1004_bedroom, 14: room_1004_bedroom.1, 15: room_1004_bedroom.2, 16: room_1004_bedroom.3, 17: room_1004_bedroom.4, 18: room_1005_bedroom, 19: room_1005_bedroom.1, 20: room_1005_bedroom.2, 21: room_1005_bedroom.3, 22: room_1005_bedroom.4, 23: room_1006_bedroom, 24: room_1006_bedroom.1, 25: room_1006_bedroom.2, 26: room_1006_bedroom.3, 27: room_1007_bedroom, 28: room_1007_bedroom.1, 29: room_1007_bedroom.2, 30: room_1007_bedroom.3, 31: room_1007_bedroom.4, 32: room_1008_bedroom, 33: room_1008_bedroom.1, 34: room_1008_bedroom.2, 35: room_1008_bedroom.3, 36: room_1008_bedroom

,device_id,room_1001_bedroom,room_1001_bedroom.1,room_1001_bedroom.2,room_1002_bedroom,room_1002_bedroom.1,room_1002_bedroom.2,room_1002_bedroom.3,room_1002_bedroom.4,room_1003_bedroom,...,room_937_bedroom.4,room_1612_bedroom.4,room_1830_bedroom.4,room_2323_bedroom.4,room_2323_living_room.4,room_817_bedroom.4,room_905_bedroom.4,room_928_bedroom.4,room_939_bedroom.4,room_1011_bedroom.4
0,datapoint,motion,presence_state,room_temperature,co2,motion,presence_state,relative_humidity,room_temperature,co2,...,relative_humidity,motion,motion,motion,motion,motion,motion,motion,motion,motion
1,datetime,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-12-01 00:00:00+07:00,0.0,0.0,23.0,728.0,1.0,1.0,64.0,21.84499931335449,400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-12-01 00:05:00+07:00,0.0,0.0,23.0,736.0,0.0,1.0,67.0,21.61000061035156,420.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-12-01 00:10:00+07:00,0.0,0.0,23.0,753.0,0.0,1.0,66.0,21.780000686645508,460.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Extract sensor row & prepare timestamp

In [13]:
sensor_row = raw.iloc[0]
data = raw.iloc[2:].copy()

data = data.rename(columns={"device_id": "timestamp"})
data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")

try:
    data["timestamp"] = data["timestamp"].dt.tz_localize(None)
except Exception:
    pass

data.head()

,timestamp,room_1001_bedroom,room_1001_bedroom.1,room_1001_bedroom.2,room_1002_bedroom,room_1002_bedroom.1,room_1002_bedroom.2,room_1002_bedroom.3,room_1002_bedroom.4,room_1003_bedroom,...,room_937_bedroom.4,room_1612_bedroom.4,room_1830_bedroom.4,room_2323_bedroom.4,room_2323_living_room.4,room_817_bedroom.4,room_905_bedroom.4,room_928_bedroom.4,room_939_bedroom.4,room_1011_bedroom.4
2,2025-12-01 00:00:00,0.0,0.0,23.0,728.0,1.0,1.0,64.0,21.84499931335449,400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-12-01 00:05:00,0.0,0.0,23.0,736.0,0.0,1.0,67.0,21.61000061035156,420.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-12-01 00:10:00,0.0,0.0,23.0,753.0,0.0,1.0,66.0,21.780000686645508,460.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2025-12-01 00:15:00,0.0,0.0,23.0,761.0,0.0,1.0,65.0,21.875,419.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2025-12-01 00:20:00,0.0,0.0,23.0,751.0,0.0,1.0,65.0,21.825000762939453,412.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Build sensor-column mappings

In [14]:
sensor_name_map = {
    "relative_humidity": "RH",
    "co2": "CO2",
    "room_temperature": "temp",
    "motion": "Motion",
    "presence_state": "Presence",
}

mappings = []
for col in raw.columns:
    if col == "device_id":
        continue
    sensor_raw = str(sensor_row[col]).strip()
    if sensor_raw in sensor_name_map:
        mappings.append((
            col,                      # wide_col
            col.split(".")[0],        # room_area
            sensor_name_map[sensor_raw]  # original_sensor_type
        ))

print(f"found sensor columns: {len(mappings)}")

found sensor columns: 2404


## 4. Wide → Long transformation

In [15]:
parts = []
ts = data["timestamp"]

for wide_col, room_area, original_sensor_type in mappings:
    parts.append(pd.DataFrame({
        "timestamp": ts,
        "room_area": room_area,
        "original_sensor_type": original_sensor_type,
        "value": pd.to_numeric(data[wide_col], errors="coerce").astype("float32")
    }))

long_df = pd.concat(parts, ignore_index=True)
long_df = long_df.dropna(subset=["timestamp"]).reset_index(drop=True)

long_df["room_area"] = long_df["room_area"].astype("category")
long_df["original_sensor_type"] = long_df["original_sensor_type"].astype("category")

del raw, data, parts, ts, sensor_row
gc.collect()

print(long_df.shape)
long_df.head()

(53311104, 4)


,timestamp,room_area,original_sensor_type,value
0,2025-12-01 00:00:00,room_1001_bedroom,Motion,0.0
1,2025-12-01 00:05:00,room_1001_bedroom,Motion,0.0
2,2025-12-01 00:10:00,room_1001_bedroom,Motion,0.0
3,2025-12-01 00:15:00,room_1001_bedroom,Motion,0.0
4,2025-12-01 00:20:00,room_1001_bedroom,Motion,0.0


In [16]:
long_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 53311104 entries, 0 to 53311103
Data columns (total 4 columns):
 #   Column                Dtype         
---  ------                -----         
 0   timestamp             datetime64[us]
 1   room_area             category      
 2   original_sensor_type  category      
 3   value                 float32       
dtypes: category(2), datetime64[us](1), float32(1)
memory usage: 762.7 MB


## 5. Per-sensor threshold validation

In [17]:
v = long_df["value"]
is_nan = v.isna()

long_df["RH_status"] = np.select(
    [is_nan, v.between(35, 150, inclusive="both")],
    ["nan", "valid"],
    default="invalid"
)

long_df["CO2_status"] = np.select(
    [is_nan, v.between(200, 5000, inclusive="both")],
    ["nan", "valid"],
    default="invalid"
)

long_df["temp_status"] = np.select(
    [is_nan, v.between(15, 30, inclusive="both")],
    ["nan", "valid"],
    default="invalid"
)

long_df["Motion_status"] = np.select(
    [is_nan, v.isin([0, 1])],
    ["nan", "valid"],
    default="invalid"
)

long_df["Presence_status"] = np.select(
    [is_nan, v.isin([0, 1, 3, 4])],
    ["nan", "valid"],
    default="invalid"
)

for c in ["RH_status", "CO2_status", "temp_status", "Motion_status", "Presence_status"]:
    long_df[c] = long_df[c].astype("category")

del v, is_nan
gc.collect()

print("Threshold validation done.")

Threshold validation done.


## 6. Count valid candidates & identify single-candidate type

In [18]:
rh_valid = long_df["RH_status"].eq("valid")
co2_valid = long_df["CO2_status"].eq("valid")
temp_valid = long_df["temp_status"].eq("valid")
motion_valid = long_df["Motion_status"].eq("valid")
presence_valid = long_df["Presence_status"].eq("valid")

long_df["n_candidates"] = (
    rh_valid.astype("int8")
    + co2_valid.astype("int8")
    + temp_valid.astype("int8")
    + motion_valid.astype("int8")
    + presence_valid.astype("int8")
).astype("int8")

# NOTE: use empty string as default (NumPy 2.x does not allow mixing str + float)
long_df["single_candidate_type"] = np.select(
    [
        long_df["n_candidates"].eq(1) & rh_valid,
        long_df["n_candidates"].eq(1) & co2_valid,
        long_df["n_candidates"].eq(1) & temp_valid,
        long_df["n_candidates"].eq(1) & motion_valid,
        long_df["n_candidates"].eq(1) & presence_valid,
    ],
    ["RH", "CO2", "temp", "Motion", "Presence"],
    default=""
)
long_df.loc[long_df["single_candidate_type"] == "", "single_candidate_type"] = np.nan

long_df["single_candidate_type"] = long_df["single_candidate_type"].astype("category")

gc.collect()

long_df["n_candidates"].value_counts()

n_candidates
1    33266462
2    19891940
0      152702
Name: count, dtype: int64

## 7. Quality flag assignment

In [19]:
orig = long_df["original_sensor_type"]

original_is_valid = (
    ((orig == "RH") & rh_valid) |
    ((orig == "CO2") & co2_valid) |
    ((orig == "temp") & temp_valid) |
    ((orig == "Motion") & motion_valid) |
    ((orig == "Presence") & presence_valid)
)

long_df["quality_flag"] = np.select(
    [
        long_df["value"].isna(),
        long_df["n_candidates"].eq(0),
        long_df["n_candidates"].eq(1) & original_is_valid,
        long_df["n_candidates"].eq(1) & (~original_is_valid),
        long_df["n_candidates"].gt(1) & original_is_valid,
        long_df["n_candidates"].gt(1) & (~original_is_valid),
    ],
    [
        "missing",
        "invalid_all_thresholds",
        "matched_original",
        "single_candidate_misplaced",
        "multi_candidate_keep_original",
        "multi_candidate_ambiguous",
    ],
    default="unknown"
)

long_df["quality_flag"] = long_df["quality_flag"].astype("category")

del orig, original_is_valid
gc.collect()

long_df["quality_flag"].value_counts(dropna=False)

quality_flag
single_candidate_misplaced       23900221
multi_candidate_ambiguous        10482019
multi_candidate_keep_original     9409921
matched_original                  9366241
invalid_all_thresholds             148078
missing                              4624
Name: count, dtype: int64

## 8. Resolve sensor type & stage-1 cleaning

In [20]:
# NOTE: use empty string as default (NumPy 2.x does not allow mixing str + float)
long_df["resolved_sensor_type"] = np.select(
    [
        long_df["quality_flag"].eq("matched_original"),
        long_df["quality_flag"].eq("single_candidate_misplaced"),
        long_df["quality_flag"].eq("multi_candidate_keep_original"),
    ],
    [
        long_df["original_sensor_type"].astype(str),
        long_df["single_candidate_type"].astype(str),
        long_df["original_sensor_type"].astype(str),
    ],
    default=""
)
long_df.loc[long_df["resolved_sensor_type"] == "", "resolved_sensor_type"] = np.nan

long_df["resolved_sensor_type"] = long_df["resolved_sensor_type"].astype("category")

long_df["cleaned_value_stage1"] = long_df["value"].astype("float32")

mask_null = long_df["quality_flag"].isin([
    "missing",
    "invalid_all_thresholds",
    "multi_candidate_ambiguous"
])

long_df.loc[mask_null, "cleaned_value_stage1"] = np.nan

gc.collect()

# Drop per-sensor status columns (no longer needed)
long_df.drop(
    columns=["RH_status", "CO2_status", "temp_status", "Motion_status", "Presence_status"],
    inplace=True
)
gc.collect()

print("Stage-1 cleaning done.")
long_df.head()

Stage-1 cleaning done.


,timestamp,room_area,original_sensor_type,value,n_candidates,single_candidate_type,quality_flag,resolved_sensor_type,cleaned_value_stage1
0,2025-12-01 00:00:00,room_1001_bedroom,Motion,0.0,2,NaN,multi_candidate_keep_original,Motion,0.0
1,2025-12-01 00:05:00,room_1001_bedroom,Motion,0.0,2,NaN,multi_candidate_keep_original,Motion,0.0
2,2025-12-01 00:10:00,room_1001_bedroom,Motion,0.0,2,NaN,multi_candidate_keep_original,Motion,0.0
3,2025-12-01 00:15:00,room_1001_bedroom,Motion,0.0,2,NaN,multi_candidate_keep_original,Motion,0.0
4,2025-12-01 00:20:00,room_1001_bedroom,Motion,0.0,2,NaN,multi_candidate_keep_original,Motion,0.0


## 9. Rolling MAD outlier detection (stage 2)

In [21]:
def add_rolling_mad_flags_lowram(df, value_col="cleaned_value_stage1", window=11, mad_multiplier=5):
    df = df.sort_values(["resolved_sensor_type", "room_area", "timestamp"]).copy()
    df["outlier_flag"] = False

    for sensor in ["RH", "CO2", "temp"]:
        idx = df.index[df["resolved_sensor_type"].eq(sensor)]
        if len(idx) == 0:
            continue

        sub = df.loc[idx, ["room_area", value_col]].copy()

        rolling_median = sub.groupby("room_area")[value_col].transform(
            lambda s: s.rolling(window=window, center=True, min_periods=3).median()
        )

        abs_dev = (sub[value_col] - rolling_median).abs()

        rolling_mad = abs_dev.groupby(sub["room_area"]).transform(
            lambda s: s.rolling(window=window, center=True, min_periods=3).median()
        )

        outlier = abs_dev > (mad_multiplier * rolling_mad)
        df.loc[idx, "outlier_flag"] = outlier.fillna(False).to_numpy()

        del sub, rolling_median, abs_dev, rolling_mad, outlier
        gc.collect()

    return df


long_df = add_rolling_mad_flags_lowram(
    long_df,
    value_col="cleaned_value_stage1",
    window=11,
    mad_multiplier=5
)

gc.collect()

# Null out outliers for continuous sensors
long_df["cleaned_value_stage2"] = long_df["cleaned_value_stage1"].astype("float32")

cont_mask = long_df["resolved_sensor_type"].isin(["RH", "CO2", "temp"])
long_df.loc[cont_mask & long_df["outlier_flag"], "cleaned_value_stage2"] = np.nan

gc.collect()

print("Outlier detection done.")
long_df.groupby("resolved_sensor_type", observed=True)["outlier_flag"].sum().sort_values(ascending=False)

Outlier detection done.


resolved_sensor_type
temp        1349068
CO2          918998
RH           907295
Motion            0
Presence          0
Name: outlier_flag, dtype: int64

## 10. Impute continuous sensors (interpolate + rolling median)

In [22]:
def impute_continuous_lowram(df, value_col="cleaned_value_stage2", window=11):
    df = df.sort_values(["resolved_sensor_type", "room_area", "timestamp"]).copy()
    df["imputed_value"] = df[value_col].astype("float32")

    for sensor in ["RH", "CO2", "temp"]:
        idx = df.index[df["resolved_sensor_type"].eq(sensor)]
        if len(idx) == 0:
            continue

        sub = df.loc[idx, ["room_area", value_col]].copy()

        filled = sub.groupby("room_area")[value_col].transform(
            lambda s: s.interpolate(limit=3, limit_direction="both")
                      .fillna(s.rolling(window, center=True, min_periods=3).median())
        )

        df.loc[idx, "imputed_value"] = filled.astype("float32").to_numpy()

        del sub, filled
        gc.collect()

    return df


long_df = impute_continuous_lowram(long_df, value_col="cleaned_value_stage2", window=11)

gc.collect()

print("Continuous imputation done.")

Continuous imputation done.


## 11. Impute discrete sensors (ffill / bfill)

In [23]:
def impute_discrete_lowram(df, value_col="imputed_value"):
    df = df.sort_values(["resolved_sensor_type", "room_area", "timestamp"]).copy()

    for sensor in ["Motion", "Presence"]:
        idx = df.index[df["resolved_sensor_type"].eq(sensor)]
        if len(idx) == 0:
            continue

        sub = df.loc[idx, ["room_area", value_col]].copy()

        filled = sub.groupby("room_area")[value_col].transform(
            lambda s: s.ffill().bfill()
        )

        df.loc[idx, value_col] = filled.astype("float32").to_numpy()

        del sub, filled
        gc.collect()

    return df


long_df = impute_discrete_lowram(long_df, value_col="imputed_value")

gc.collect()

print("Discrete imputation done.")

Discrete imputation done.


## 12. Final output & summary statistics

In [24]:
final_df = long_df[[
    "timestamp",
    "room_area",
    "original_sensor_type",
    "resolved_sensor_type",
    "value",
    "quality_flag",
    "cleaned_value_stage1",
    "outlier_flag",
    "cleaned_value_stage2",
    "imputed_value",
    "n_candidates",
    "single_candidate_type",
]].copy()

final_df.head(20)

,timestamp,room_area,original_sensor_type,resolved_sensor_type,value,quality_flag,cleaned_value_stage1,outlier_flag,cleaned_value_stage2,imputed_value,n_candidates,single_candidate_type
50539104,2025-12-01 00:00:00,room_1001_bedroom,CO2,CO2,502.0,matched_original,502.0,False,502.0,502.0,1,CO2
50539105,2025-12-01 00:05:00,room_1001_bedroom,CO2,CO2,504.0,matched_original,504.0,False,504.0,504.0,1,CO2
50539106,2025-12-01 00:10:00,room_1001_bedroom,CO2,CO2,506.0,matched_original,506.0,False,506.0,506.0,1,CO2
50539107,2025-12-01 00:15:00,room_1001_bedroom,CO2,CO2,516.0,matched_original,516.0,False,516.0,516.0,1,CO2
50539108,2025-12-01 00:20:00,room_1001_bedroom,CO2,CO2,510.0,matched_original,510.0,False,510.0,510.0,1,CO2
50539109,2025-12-01 00:25:00,room_1001_bedroom,CO2,CO2,509.0,matched_original,509.0,False,509.0,509.0,1,CO2
50539110,2025-12-01 00:30:00,room_1001_bedroom,CO2,CO2,511.0,matched_original,511.0,False,511.0,511.0,1,CO2
50539111,2025-12-01 00:35:00,room_1001_bedroom,CO2,CO2,510.0,matched_original,510.0,False,510.0,510.0,1,CO2
50539112,2025-12-01 00:40:00,room_1001_bedroom,CO2,CO2,512.0,matched_original,512.0,False,512.0,512.0,1,CO2
50539113,2025-12-01 00:45:00,room_1001_bedroom,CO2,CO2,515.0,matched_original,515.0,False,515.0,515.0,1,CO2


In [25]:
final_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
Index: 53311104 entries, 50539104 to 51905952
Data columns (total 12 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   timestamp              datetime64[us]
 1   room_area              category      
 2   original_sensor_type   category      
 3   resolved_sensor_type   category      
 4   value                  float32       
 5   quality_flag           category      
 6   cleaned_value_stage1   float32       
 7   outlier_flag           bool          
 8   cleaned_value_stage2   float32       
 9   imputed_value          float32       
 10  n_candidates           int8          
 11  single_candidate_type  category      
dtypes: bool(1), category(5), datetime64[us](1), float32(4), int8(1)
memory usage: 4.0 GB


In [26]:
final_df["quality_flag"].value_counts(dropna=False)

quality_flag
single_candidate_misplaced       23900221
multi_candidate_ambiguous        10482019
multi_candidate_keep_original     9409921
matched_original                  9366241
invalid_all_thresholds             148078
missing                              4624
Name: count, dtype: int64

In [27]:
final_df.groupby("resolved_sensor_type", observed=True)["outlier_flag"] \
    .sum().sort_values(ascending=False)

resolved_sensor_type
temp        1349068
CO2          918998
RH           907295
Motion            0
Presence          0
Name: outlier_flag, dtype: int64

In [28]:
final_df.groupby("resolved_sensor_type", observed=True)["cleaned_value_stage2"] \
    .apply(lambda s: s.isna().mean() * 100) \
    .sort_values(ascending=False)

resolved_sensor_type
temp        12.595236
CO2          8.602045
RH           8.496019
Motion       0.000000
Presence     0.000000
Name: cleaned_value_stage2, dtype: float64

## 13. Save to CSV

In [29]:
final_df.to_csv(save_path, index=False)
print("saved to:", save_path)

saved to: /Users/phasurab/Desktop/Alto_test/final_cleaned_phase1.csv
